## A2A Client to Interact with Our Server

In [4]:
%pip install a2a-sdk==0.3.8 azure-ai-projects==2.0.0b2 python-dotenv azure-identity uvicorn

Note: you may need to restart the kernel to use updated packages.


### Importing Libraries and Utilities

In [5]:
import logging

from typing import Any
from uuid import uuid4

import httpx

from a2a.client import A2ACardResolver, A2AClient
from a2a.types import (
    AgentCard,
    MessageSendParams,
    SendMessageRequest,
    SendStreamingMessageRequest,
)

### A2A Client Setup and Implementation

In [6]:
import uuid
from httpx import AsyncClient
from a2a.types import AgentCard, Message, Part, Role, Task, TextPart
from a2a.client import ClientFactory, Client
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.types import TransportProtocol
from a2a.client import create_text_message_object
from a2a.utils.message import get_message_text

base_url = "http://localhost:8080"

# creating the httpx client with custom timeouts
async with httpx.AsyncClient(
    timeout=httpx.Timeout(
        connect=10.0,
        read=60.0,
        write=10.0,
        pool=10.0,
            )) as httpx_client:
    
    # creating the A2ACardResolver
    resolver = A2ACardResolver(
        httpx_client=httpx_client,
        base_url=base_url
    )

    final_agent_card_to_use: AgentCard | None = None

    try:
        # Attempting to fetch public agent card from custom path
        _public_card = (
                await resolver.get_agent_card()
            )  # Fetches from default public path

        final_agent_card_to_use = _public_card

    except Exception as e:
        print(f'Critical error fetching public agent card: {e}', exc_info=True)  
        raise RuntimeError(
                'Failed to fetch the public agent card. Cannot continue.'
            ) from e

    # initializing the A2A Client (ClientFactory Class) config
    config = ClientConfig(
        httpx_client = httpx_client,
        supported_transports=[
                    TransportProtocol.jsonrpc
        ],
        streaming=True
    )
    
    # Creating the A2A Client - Client Factory
    client = ClientFactory(config).create(final_agent_card_to_use)
    
    # Creating the message request
    request = create_text_message_object(content="Tell me how to create an Azure Storage Account using Azure CLI.")

    # Send the request and get the streaming messages
    async for response in client.send_message(request):
                task, _ = response
                print(get_message_text(task.artifacts[-1]), end = "", flush = True)

Here’s a concise, step-by-step guide to create an Azure Storage account with Azure CLI.

Prerequisites
- Install the Azure CLI (or use Azure Cloud Shell).
- Sign in: az login
- Storage account names must be globally unique, lowercase letters and numbers, 3–24 characters.

Steps and example commands

1) (Optional) create a resource group (if you don’t already have one)
az group create --name myResourceGroup --location eastus

2) (Optional) list available regions
az account list-locations --query "[].{Region:name}" --out table

3) Create a general-purpose v2 storage account (common example)
Replace <account-name>, <resource-group>, and <location> with your values:
az storage account create \
  --name <account-name> \
  --resource-group <resource-group> \
  --location <location> \
  --sku Standard_RAGRS \
  --kind StorageV2 \
  --min-tls-version TLS1_2 \
  --allow-blob-public-access false

Notes on commonly used options
- --sku: Standard_LRS, Standard_GRS, Standard_RAGRS, Standard_ZRS, Pr